# Plant Disease Detection — ResNet50

This notebook runs the ResNet50 experiment cell by cell for the SE4050 project. It covers dataset inspection, reproducible splitting, ImageNet preprocessing, two-stage transfer learning, and final evaluation. The test split is not used until the final evaluation cell.

Dataset: local PlantVillage copy. The dataset is not committed to Git.

In [ ]:
from pathlib import Path
import json, subprocess, sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path('/Users/thakkifilyas/Desktop/DL_Project/PlantVillage')
CONFIG = PROJECT_ROOT / 'configs' / 'resnet50_config.yaml'
RESULTS = PROJECT_ROOT / 'results' / 'resnet50'
assert DATA_DIR.is_dir(), f'Dataset not found: {DATA_DIR}'
print('Project:', PROJECT_ROOT)
print('Dataset:', DATA_DIR)

## 1. Dataset inspection / EDA
This cell checks class counts, image integrity, dimensions, imbalance, and representative samples.

In [ ]:
subprocess.run([sys.executable, '-m', 'resnet50.eda', '--config', str(CONFIG)], cwd=PROJECT_ROOT, check=True)

In [ ]:
summary = json.loads((RESULTS / 'eda' / 'summary.json').read_text())
summary

## 2. Reproducible split
The reusable splitter creates a stratified 70/15/15 manifest with seed 42. Use the same manifest for every group model after the group agrees on it.

In [ ]:
import yaml
SHARED_MANIFEST = Path('/tmp/resnet50_shared_split.csv')
subprocess.run([sys.executable, '-m', 'resnet50.split', '--data-dir', str(DATA_DIR), '--output', str(SHARED_MANIFEST), '--seed', '42'], cwd=PROJECT_ROOT, check=True)
runtime_config = yaml.safe_load(CONFIG.read_text()); runtime_config['data_dir'] = str(DATA_DIR); runtime_config['manifest'] = str(SHARED_MANIFEST)
CONFIG = Path('/tmp/resnet50_notebook_config.yaml'); CONFIG.write_text(yaml.safe_dump(runtime_config))
print(SHARED_MANIFEST)

## 3. Train ResNet50
Stage A freezes the ImageNet backbone. Stage B fine-tunes `conv5_block1` and later layers with a smaller learning rate. Training augmentation is enabled; validation and test transformations remain deterministic. This cell can take a long time on CPU.

In [ ]:
subprocess.run([sys.executable, '-m', 'resnet50.train', '--config', str(CONFIG)], cwd=PROJECT_ROOT, check=True)

In [ ]:
import csv
with open(RESULTS / 'training_metrics.csv', newline='') as f:
    training_metrics = list(csv.DictReader(f))
training_metrics[-5:]

## 4. Final evaluation
Run this only after model development is complete. It evaluates the selected checkpoint once on the untouched test split and creates the confusion matrix, classification report, per-class metrics, ROC-AUC, curves, parameter counts, and timing.

In [ ]:
CHECKPOINT = RESULTS / 'best.keras'
assert CHECKPOINT.exists(), 'Train the model first'
subprocess.run([sys.executable, '-m', 'resnet50.evaluate', '--config', str(CONFIG), '--checkpoint', str(CHECKPOINT)], cwd=PROJECT_ROOT, check=True)

In [ ]:
metrics = json.loads((RESULTS / 'evaluation' / 'final_metrics.json').read_text())
print('Test accuracy:', metrics['test_accuracy'])
print('Weighted F1:', metrics['test_f1_weighted'])
print('ROC-AUC:', metrics.get('test_roc_auc_ovr_weighted'))
print('Parameters:', metrics['total_parameters'])
print('Inference seconds/image:', metrics['inference_seconds_per_image'])

In [ ]:
from IPython.display import Image, display
evaluation_dir = RESULTS / 'evaluation'
display(Image(filename=str(evaluation_dir / 'accuracy_curve.png')))
display(Image(filename=str(evaluation_dir / 'loss_curve.png')))
display(Image(filename=str(evaluation_dir / 'confusion_matrix.png')))
display(Image(filename=str(evaluation_dir / 'per_class_performance.png')))

## Reproducibility notes
- Seed: 42
- Input: 224×224 RGB
- ImageNet preprocessing is applied inside the model.
- Only training data is augmented.
- Validation controls early stopping and checkpoint selection.
- The test set is used only in the final evaluation cell.
- Checkpoints are local artifacts and are excluded from Git.